> **Mask pipeline archived (§1–§9).** Segmentation and mask generation have already been
> run; the masks live under `Results/masks/<image>/*.npy` and the per-cell analysis in
> `Results/L3 Lipophagy images_LC3_lipophagy_mitophagy_analysis.csv`. Those cells are commented
> out (real code, `#`-prefixed) rather than deleted or left executable — this notebook is being
> uploaded separately as the reference implementation for mask generation, and they need
> `czifile` and `cellpose`, which are not installed here.
>
> **§10 is the only live cell.** It reads the saved masks and writes the additional-analysis
> CSVs. It is self-contained (own imports and paths), so it runs on its own.

Red = lipid droplets · Yellow = mutant LC3 · Green = normal LC3 · Blue = mitochondria

Segment whole cells from the yellow+green composite, then classify green-positive cells. For LC3 structures, count each dot/ring separately; fill hollow rings before overlap tests so an encased red or blue object still counts as a match. MFI is measured on raw channels only.


## Imports
Download all dependencies prior to running

In [1]:
# %matplotlib inline
# import os
# os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
#
# import glob
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import matplotlib.patches as mpatches
# from matplotlib.colors import LinearSegmentedColormap, ListedColormap
#
# import czifile
# from scipy import ndimage as ndi
# from skimage import exposure, morphology
# from skimage.color import label2rgb
# from skimage.filters import threshold_otsu, gaussian
# from skimage.measure import label, regionprops
# from skimage.morphology import remove_small_objects, disk, binary_dilation, closing
# from skimage.segmentation import relabel_sequential, watershed

## 2 · Parameters & segmentation method

In [2]:
# BASE_DIR = "."
# IMAGE_FOLDER_NAMES = [
#     "L3 Lipophagy images",
# ]
#
#
# def resolve_image_folder(name):
#     """Return the first existing path for `name` (under BASE_DIR, then images/),
#     else the BASE_DIR path (which will simply be skipped if it never appears)."""
#     for cand in (os.path.join(BASE_DIR, name), os.path.join(BASE_DIR, "images", name)):
#         if os.path.isdir(cand):
#             return cand
#     return os.path.join(BASE_DIR, name)
#
#
# IMAGE_FOLDERS = [resolve_image_folder(n) for n in IMAGE_FOLDER_NAMES]
#
# FILE_EXTENSION = "*.czi"
# RESULTS_FOLDER = "Results"
# MASK_FOLDER = os.path.join(RESULTS_FOLDER, "masks")
# CSV_SUFFIX = "LC3_lipophagy_mitophagy_analysis.csv"  # CSV name = "<folder>_<CSV_SUFFIX>"
# SHOW_PLOTS = True  # False skips the QC panels
#
#
# CHANNEL_ORDER = {"red": 0, "yellow": 1, "green": 2, "blue": 3}
#
# USE_CELLPOSE = True
# USE_GPU = True
#
# # Cell segmentation: Cellpose cyto3 on the yellow+green composite.
# CYTO_SEG_CHANNELS = ["yellow", "green"]
# CYTO_SMOOTH_SIGMA = 3.0 # pre-blur to suppress texture
# CYTO_DIAMETER = 250
# CYTO_FLOW_THRESH = 0.9 # higher = more lenient
# CYTO_PROB_THRESH = -1.0 # lower = more lenient
# CELL_MIN_AREA = 3000 
# CELL_MIN_MEAN_SIGNAL = 0.12 # min mean composite for a label to be kept
# CELL_MERGE_BORDER_FRAC = 0.75 # merge two labels if their shared border is >= this * the
#                               # dimmer cell's mean composite
# CELL_MERGE_MIN_BORDER = 50    # ...and they share at least this many border px
# CELL_MIN_MEAN_SIGNAL_DIM = 0.05 # pass-2 gate; background sits around 0.03
# CELL_DIM_HALO = 8               # px excluded around pass-1 cells before pass 2
#
#
# GREEN_FILL_SIGMA = 2.0
# GREEN_FILL_THRESH = 0.08 # percentile-normed green above this counts as green
# GREEN_FILL_FRAC = 0.40 # green-positive if >= this fraction of the cell is green
#
# # Normal-LC3 structures (green dots + rings). Top-hat above the diffuse LC3-I cytoplasm,
# # then a per-cell adaptive threshold; rings are sealed per component and touching objects
# # are split.
# GREEN_TOPHAT_RADIUS = 14     # >= radius of the largest ring to capture whole
# GREEN_PRESMOOTH = 0.6       # bridges micro-gaps in uneven ring walls
# GREEN_TOPHAT_THRESH = 0.0035
# GREEN_MIN_BRIGHTNESS = 0.03
# GREEN_LOCAL_ZSCORE = 1.0    # local threshold = mean + z*std of the in-cell top-hat
# GREEN_LOCAL_PCTL = 88       # ...or this percentile of it, whichever is higher
# GREEN_PROM_MULT = 4.0       # ...or this * its median. Diffuse noCH cells never clear this,
#                             # so they yield no phantom puncta.
# GREEN_MIN_SIZE = 10
# GREEN_MAX_SIZE = 4000
# GREEN_RING_CLOSE = 3        # per-component closing radius for broken ring walls
# GREEN_SPLIT_H = 2.0          # h-maxima depth for the splitting watershed
# GREEN_RING_MIN_AREA = 45    # min component area for circle reconstruction
# GREEN_RING_MAX_ECC = 0.995
# GREEN_CIRC_RESID = 2.2      # max RMS radial residual (px) to accept an arc as a ring
# GREEN_CIRC_MIN_R = 4
# GREEN_CIRC_MAX_R = 60
# GREEN_CIRC_MIN_COVER = 0.30 # min fraction of the circumference the arc must span
# GREEN_CLOSE_RADIUS = 1      # legacy, used only by the superseded §4 fill_green_rings
#
# # Lipid droplets (red): bright dots on a diffuse LipidTOX background.
# RED_TOPHAT_RADIUS = 10
# RED_PCTL_THRESH = 99.3
# RED_MIN_SIZE = 6
# RED_MAX_SIZE = 2000
#
# # Mitochondria (blue): white top-hat on the mito-BFP channel.
# BLUE_SMOOTH_SIGMA = 1.0     # legacy, used only by the superseded §4 detect_mitochondria
# BLUE_TOPHAT_RADIUS = 6      # >= half a streak's width; strips the wider-scale haze
# BLUE_OTSU_FACTOR = 0.7      # threshold = max(otsu * factor, floor)
# BLUE_FLOOR = 0.05
# BLUE_MIN_SIZE = 40          # drop sub-streak specks
#
# SEG_SMOOTH_SIGMA = 10
# SEG_FG_FLOOR = 0.05
#
# os.makedirs(RESULTS_FOLDER, exist_ok=True)
# os.makedirs(MASK_FOLDER, exist_ok=True)
#
# cyto_model = None
# if USE_CELLPOSE:
#     from cellpose import models
#     print("Initializing Cellpose (cyto3)...")
#     cyto_model = models.Cellpose(gpu=USE_GPU, model_type="cyto3")
#     print(f"  cyto3 ready  |  GPU={USE_GPU}")
# else:
#     print("Using intensity-based segmentation.")
# print(f"Parameters set. {len(IMAGE_FOLDERS)} folder(s) queued.")

## 3 · Loading & normalization

In [3]:
# def normalize_image(image):
#     """Min-max normalize to float64 [0, 1]. Used for the inclusion analysis."""
#     img = image.astype(np.float64)
#     mn, mx = img.min(), img.max()
#     return np.zeros_like(img) if mx == mn else (img - mn) / (mx - mn)
#
#
# def pct_norm(image, lo=1.0, hi=99.5):
#     """Percentile contrast-stretch to float32 [0, 1]. Used for segmentation.
#
#     Min-max is unusable here: single saturated pixels (raw max ~65535) crush the real
#     signal to ~0.01, leaving Cellpose nothing to work with."""
#     image = image.astype(np.float32)
#     p_lo, p_hi = np.percentile(image, (lo, hi))
#     if p_hi <= p_lo:
#         return np.zeros_like(image)
#     return np.clip((image - p_lo) / (p_hi - p_lo), 0, 1)
#
#
# def boost_contrast_visual(image):
#     """Percentile stretch, for display only."""
#     p_lo, p_hi = np.percentile(image, (1, 99.5))
#     return exposure.rescale_intensity(image, in_range=(p_lo, p_hi))
#
#
# def load_four_channels(filepath):
#     """Load 4-channel .czi -> dict {'red','yellow','green','blue'} per CHANNEL_ORDER."""
#     try:
#         czi = czifile.CziFile(filepath)
#         img = np.squeeze(czi.asarray())
#     except Exception as e:
#         print(f"  [ERROR] File load failed: {e}")
#         return None
#     if img.ndim == 3 and 4 in img.shape:
#         ch_axis = int(np.where(np.array(img.shape) == 4)[0][0])
#         img = np.moveaxis(img, ch_axis, 0)
#     else:
#         print(f"  [ERROR] Expected a 4-channel image, got shape {img.shape}")
#         return None
#     return {name: img[idx] for name, idx in CHANNEL_ORDER.items()}

## 4 · Mask functions (cells, green‑positive, green dots+rings, lipid droplets, mitochondria)

In [4]:
#
# # Cell segmentation and green-positive classification. Runs on the percentile-normalized
# # `seg` dict throughout.
#
# def drop_small_labels(labels, min_area):
#     """Zero out labeled regions smaller than min_area, then relabel 1..N."""
#     labels = labels.astype(np.int32)
#     counts = np.bincount(labels.ravel())
#     small = np.flatnonzero(counts < min_area)
#     small = small[small != 0]
#     if small.size:
#         labels[np.isin(labels, small)] = 0
#     return relabel_sequential(labels)[0]
#
#
# def cyto_composite(seg):
#     """Whole-cell signal: pixel-wise max of CYTO_SEG_CHANNELS."""
#     return np.maximum.reduce([seg[c] for c in CYTO_SEG_CHANNELS])
#
#
# def merge_false_splits(labels, composite):
#     """Merge two adjacent labels when their shared border runs through bright tissue.
#
#     A real cell-cell boundary has a dim membrane valley; a phantom split through one cell
#     does not. Spatially separate objects share no border and are always kept."""
#     labels = labels.astype(np.int32)
#     ids = [p.label for p in regionprops(labels)]
#     if len(ids) < 2:
#         return labels
#     interior = {p.label: float(composite[labels == p.label].mean()) for p in regionprops(labels)}
#     parent = {i: i for i in ids}
#     def find(x):
#         while parent[x] != x:
#             parent[x] = parent[parent[x]]; x = parent[x]
#         return x
#     for a in range(len(ids)):
#         for b in range(a + 1, len(ids)):
#             ma, mb = labels == ids[a], labels == ids[b]
#             border = (ndi.binary_dilation(ma) & mb) | (ndi.binary_dilation(mb) & ma)
#             if int(border.sum()) < CELL_MERGE_MIN_BORDER:
#                 continue
#             if float(composite[border].mean()) >= CELL_MERGE_BORDER_FRAC * min(interior[ids[a]], interior[ids[b]]):
#                 parent[find(ids[a])] = find(ids[b])
#     out = np.zeros_like(labels)
#     remap = {}; n = 0
#     for i in ids:
#         r = find(i)
#         if r not in remap:
#             n += 1; remap[r] = n
#     for i in ids:
#         out[labels == i] = remap[find(i)]
#     for lab in range(1, n + 1):        # seal any thin watershed line between merged pieces
#         out[ndi.binary_fill_holes(out == lab)] = lab
#     return relabel_sequential(out)[0]
#
#
# def _cellpose_masks(image):
#     """Run Cellpose cyto3 on a prepared single-channel image -> int32 label map."""
#     masks, _, _, _ = cyto_model.eval(
#         image, diameter=CYTO_DIAMETER, channels=[0, 0],
#         flow_threshold=CYTO_FLOW_THRESH, cellprob_threshold=CYTO_PROB_THRESH,
#         normalize=True, resample=True,
#     )
#     return masks.astype(np.int32)
#
#
# def _gate_labels(masks, composite, min_signal):
#     """Drop labels smaller than CELL_MIN_AREA or dimmer than min_signal (mean composite)."""
#     out = masks.copy()
#     for p in regionprops(masks):
#         region = masks == p.label
#         if p.area < CELL_MIN_AREA or float(composite[region].mean()) < min_signal:
#             out[region] = 0
#     return out
#
#
# def segment_whole_cells_cellpose(seg):
#     """Two-pass Cellpose cyto3 on the smoothed yellow+green composite.
#
#     One very bright cell dominates the dynamic range and hides its dimmer neighbours under a
#     single global normalization. Pass 1 segments at native contrast; pass 2 blanks the pass-1
#     cells plus a CELL_DIM_HALO margin, re-stretches what is left so the dim cells set the
#     scale, and adds any non-overlapping labels under a lower signal gate."""
#     composite = cyto_composite(seg)
#     # Pass 1: bright cells at native contrast.
#     m1 = _gate_labels(_cellpose_masks(gaussian(composite, sigma=CYTO_SMOOTH_SIGMA)),
#                       composite, CELL_MIN_MEAN_SIGNAL)
#     combined = m1.copy()
#     nxt = int(m1.max())
#     # Pass 2: recover dim cells hidden under the bright cell's dynamic range.
#     leftover = composite.copy()
#     leftover[binary_dilation(m1 > 0, disk(CELL_DIM_HALO))] = 0
#     pos = leftover[leftover > 0]
#     if pos.size > 50 and float(pos.max()) > 0:
#         lo, hi = np.percentile(pos, (1.0, 99.5))
#         if hi > lo:
#             restretched = np.clip((leftover - lo) / (hi - lo), 0, 1)
#             m2 = _gate_labels(_cellpose_masks(gaussian(restretched, sigma=CYTO_SMOOTH_SIGMA)),
#                               composite, CELL_MIN_MEAN_SIGNAL_DIM)
#             for p in regionprops(m2):
#                 region = (m2 == p.label) & (m1 == 0)
#                 if int(region.sum()) >= CELL_MIN_AREA and float(composite[region].mean()) >= CELL_MIN_MEAN_SIGNAL_DIM:
#                     nxt += 1
#                     combined[region] = nxt
#     combined = merge_false_splits(combined, composite)
#     final, _, _ = relabel_sequential(combined)
#     print(f"    Cells (two-pass cyto3 bright+dim, signal-gated+merged): {final.max()}")
#     return final
#
#
# def segment_cells_intensity(seg):
#     """Intensity-based cell segmentation fallback (no Cellpose)."""
#     comp = cyto_composite(seg)
#     seed = gaussian(comp, sigma=SEG_SMOOTH_SIGMA)
#     try:
#         fg_thr = max(threshold_otsu(seed), SEG_FG_FLOOR)
#     except Exception:
#         fg_thr = SEG_FG_FLOOR
#     fg = remove_small_objects(seed > fg_thr, min_size=CELL_MIN_AREA)
#     fg = ndi.binary_fill_holes(fg)
#     final = drop_small_labels(label(fg), CELL_MIN_AREA)
#     print(f"    Cells (intensity): {final.max()}")
#     return final
#
#
# def classify_green_positive(labeled_cells, green_seg):
#     """Cells whose green-fill fraction reaches GREEN_FILL_FRAC -> (ids, fills)."""
#     gsm = gaussian(green_seg, sigma=GREEN_FILL_SIGMA)
#     green_on = gsm > GREEN_FILL_THRESH
#     green_ids, fills = [], {}
#     for cid in range(1, int(labeled_cells.max()) + 1):
#         cell_px = labeled_cells == cid
#         area = int(cell_px.sum())
#         if area == 0:
#             continue
#         fill = float((green_on & cell_px).sum()) / area
#         fills[cid] = fill
#         if fill >= GREEN_FILL_FRAC:
#             green_ids.append(cid)
#     return green_ids, fills
#
#
# # Inclusion analysis: green dots+rings, lipid droplets, mitochondria. Runs on the min-max
# # normalized `norm` dict.
#
# def detect_green_structures(green_norm, labeled_cells):
#     """Normal-LC3 structures: solid dots plus hollow ring walls, per cell.
#
#     A white top-hat isolates structures above the diffuse LC3-I cytoplasm and a per-cell
#     adaptive threshold keeps this robust across bright and dim cells. Ring centres stay
#     dark here; fill_green_rings turns them into disks."""
#     tophat = morphology.white_tophat(green_norm, morphology.disk(GREEN_TOPHAT_RADIUS))
#     candidate = ((tophat > GREEN_TOPHAT_THRESH) &
#                  (green_norm > GREEN_MIN_BRIGHTNESS) & (labeled_cells > 0))
#     candidate = remove_small_objects(candidate, min_size=GREEN_MIN_SIZE)
#     final = np.zeros_like(candidate, dtype=bool)
#     for cid in range(1, int(labeled_cells.max()) + 1):
#         cell_mask = labeled_cells == cid
#         tvals = tophat[cell_mask]
#         if tvals.size == 0:
#             continue
#         local_thr = max(GREEN_TOPHAT_THRESH,
#                         float(tvals.mean()) + GREEN_LOCAL_ZSCORE * float(tvals.std() or 1e-6),
#                         float(np.percentile(tvals, GREEN_LOCAL_PCTL)))
#         final[candidate & cell_mask & (tophat >= local_thr)] = True
#     final = remove_small_objects(final, min_size=GREEN_MIN_SIZE)
#     lbl = label(final)
#     for p in regionprops(lbl):
#         if p.area > GREEN_MAX_SIZE:
#             final[lbl == p.label] = False
#     return final
#
#
# def fill_green_rings(green_struct):
#     """Close wall gaps and fill holes so every ring becomes a solid disk.
#
#     The filled disk is the region the ring encases, which is what the counts and the
#     colocalization tests operate on."""
#     closed = morphology.closing(green_struct, morphology.disk(GREEN_CLOSE_RADIUS))
#     return ndi.binary_fill_holes(closed)
#
#
# def detect_bright_puncta(channel_norm, tophat_radius, pctl_thresh, min_size, max_size):
#     """Bright round puncta (used for RED lipid droplets)."""
#     tophat = morphology.white_tophat(channel_norm, morphology.disk(tophat_radius))
#     nonzero = tophat[tophat > 0]
#     if nonzero.size == 0:
#         return np.zeros_like(channel_norm, dtype=bool)
#     mask = tophat > np.percentile(nonzero, pctl_thresh)
#     mask = remove_small_objects(mask, min_size=min_size)
#     lbl = label(mask)
#     for p in regionprops(lbl):
#         if p.area > max_size:
#             mask[lbl == p.label] = False
#     return mask
#
#
# def detect_mitochondria(blue_norm, labeled_cells):
#     """Mitochondrial streaks: threshold the smoothed mito-BFP channel, drop specks,
#     restrict to cells."""
#     bsm = gaussian(blue_norm, sigma=BLUE_SMOOTH_SIGMA)
#     try:
#         thr = max(threshold_otsu(bsm) * BLUE_OTSU_FACTOR, BLUE_FLOOR)
#     except Exception:
#         thr = BLUE_FLOOR
#     mask = bsm > thr
#     mask = remove_small_objects(mask, min_size=BLUE_MIN_SIZE)
#     mask &= labeled_cells > 0
#     return mask
#
#
# print("Mask functions defined.")

In [5]:
# # LC3-structure and mitochondria masks. These supersede the definitions in §4: ring
# # seal+fill gains circle-fit reconstruction and watershed splitting, green detection gains a
# # prominence gate, and the mitochondria mask moves to a white top-hat. Parameters live in §2.
#
#
# def detect_green_structures(green_norm, labeled_cells):
#     """Green-LC3 wall and dot pixels, from the green channel alone.
#
#     The per-cell prominence gate (GREEN_PROM_MULT * median in-cell top-hat) rejects diffuse
#     cytoplasmic texture, so a bright but structure-less LC3-I cell yields no objects rather
#     than a field of phantom blobs. Ring centres stay dark; fill_green_rings fills them."""
#     tophat = morphology.white_tophat(green_norm, morphology.disk(GREEN_TOPHAT_RADIUS))
#     if GREEN_PRESMOOTH:
#         tophat = gaussian(tophat, GREEN_PRESMOOTH)   # connect near-broken walls
#     candidate = ((tophat > GREEN_TOPHAT_THRESH) &
#                  (green_norm > GREEN_MIN_BRIGHTNESS) & (labeled_cells > 0))
#     candidate = remove_small_objects(candidate, min_size=GREEN_MIN_SIZE)
#     final = np.zeros_like(candidate, dtype=bool)
#     for cid in range(1, int(labeled_cells.max()) + 1):
#         cell_mask = labeled_cells == cid
#         tvals = tophat[cell_mask]
#         if tvals.size == 0:
#             continue
#         local_thr = max(GREEN_TOPHAT_THRESH,
#                         float(tvals.mean()) + GREEN_LOCAL_ZSCORE * float(tvals.std() or 1e-6),
#                         float(np.percentile(tvals, GREEN_LOCAL_PCTL)),
#                         GREEN_PROM_MULT * float(np.median(tvals)))   # prominence gate
#         final[candidate & cell_mask & (tophat >= local_thr)] = True
#     final = remove_small_objects(final, min_size=GREEN_MIN_SIZE)
#     lbl = label(final)
#     for p in regionprops(lbl):
#         if p.area > GREEN_MAX_SIZE:
#             final[lbl == p.label] = False
#     return final
#
#
# def _fit_circle(rr, cc):
#     """Algebraic (Kasa) least-squares circle fit -> (r0, c0, R, rms_radial_residual)."""
#     A = np.c_[2 * cc, 2 * rr, np.ones(len(rr))]
#     sol, *_ = np.linalg.lstsq(A, cc ** 2 + rr ** 2, rcond=None)
#     c0, r0 = sol[0], sol[1]
#     R = np.sqrt(max(sol[2] + c0 ** 2 + r0 ** 2, 0.0))
#     d = np.sqrt((rr - r0) ** 2 + (cc - c0) ** 2)
#     rms = float(np.sqrt(np.mean((d - R) ** 2))) if len(rr) else 1e9
#     return r0, c0, R, rms
#
#
# def _reconstruct_broken_ring(comp):
#     """The disk implied by `comp` if it is an open arc that fits a circle well, else None."""
#     rr, cc = np.nonzero(comp)
#     if rr.size < 12:
#         return None
#     r0, c0, R, rms = _fit_circle(rr.astype(float), cc.astype(float))
#     if not (GREEN_CIRC_MIN_R <= R <= GREEN_CIRC_MAX_R) or rms > GREEN_CIRC_RESID:
#         return None
#     ang = np.arctan2(rr - r0, cc - c0)                      # angular coverage of the arc
#     cover = np.unique(np.floor(ang / (2 * np.pi / 36)).astype(int)).size / 36.0
#     if cover < GREEN_CIRC_MIN_COVER:
#         return None
#     Y, X = np.ogrid[:comp.shape[0], :comp.shape[1]]
#     return (Y - r0) ** 2 + (X - c0) ** 2 <= R ** 2
#
#
# def _split_touching(mask):
#     """Split touching objects with h-maxima seeds on the distance transform. A single disk
#     keeps one seed and stays whole. Only ever splits, never joins."""
#     dist = ndi.distance_transform_edt(mask)
#     markers = label(morphology.h_maxima(dist, GREEN_SPLIT_H))
#     if markers.max() == 0:
#         return label(mask)
#     return watershed(-dist, markers, mask=mask)
#
#
# def fill_green_rings(green_struct):
#     """Turn wall/dot pixels into individual filled objects -> (labels, ring_flag).
#
#     Neighbouring objects cannot merge: the hole-fill only touches enclosed interiors, the
#     repair pass runs per component, and the closing watershed only splits."""
#     filled = ndi.binary_fill_holes(green_struct)          # seal already-closed loops
#     ring_flag = filled & ~green_struct                    # an enclosed interior means a ring
#     lbl0 = label(green_struct)
#     selem = morphology.disk(GREEN_RING_CLOSE)
#     for p in regionprops(lbl0):
#         comp = lbl0 == p.label
#         if (filled & comp & ~green_struct).any():
#             continue                                       # already sealed
#         sealed = ndi.binary_fill_holes(morphology.closing(comp, selem))
#         gained = sealed & ~comp
#         if gained.sum() > 0.15 * comp.sum():               # the seal worked
#             filled |= sealed; ring_flag |= gained
#             continue
#         if p.area >= GREEN_RING_MIN_AREA and p.eccentricity <= GREEN_RING_MAX_ECC:
#             disk_fill = _reconstruct_broken_ring(comp)     # broken ring -> fitted disk
#             if disk_fill is not None:
#                 filled |= disk_fill; ring_flag |= (disk_fill & ~green_struct)
#     objects = _split_touching(filled)
#     objects = remove_small_objects(objects, GREEN_MIN_SIZE)
#     for p in regionprops(objects):
#         if p.area > GREEN_MAX_SIZE:
#             objects[objects == p.label] = 0
#     objects, _, _ = relabel_sequential(objects)
#     return objects, ring_flag
#
#
# def detect_mitochondria(blue_norm, labeled_cells):
#     """Mitochondrial streaks. The white top-hat keeps the thin streak core and discards the
#     diffuse BFP halo, so the mask reflects the real streak width rather than the fattened
#     one a direct threshold produces."""
#     tophat = morphology.white_tophat(blue_norm, morphology.disk(BLUE_TOPHAT_RADIUS))
#     try:
#         thr = max(threshold_otsu(tophat) * BLUE_OTSU_FACTOR, BLUE_FLOOR)
#     except Exception:
#         thr = BLUE_FLOOR
#     mask = tophat > thr
#     mask = remove_small_objects(mask, min_size=BLUE_MIN_SIZE)
#     mask &= labeled_cells > 0
#     return mask
#
#
# print("Loaded: merge-safe LC3 ring seal+fill (circle-fit reconstruction, watershed split), "
#       "diffuse-texture prominence gate, + strict top-hat mitochondria.")

## 5 · Colocalization helpers

In [6]:
# def coloc_green_with(green_filled_labeled, target_mask):
#     """Colocalization with the green structure as the counting unit.
#
#     Each filled dot or ring that overlaps or encases the target counts once, so one mito
#     streak crossing two LC3 rings counts as 2. Returns (n_structures, overlap_area_px,
#     coloc_mask), the area being filled-green n target over the colocalizing structures."""
#     n = 0
#     coloc = np.zeros(green_filled_labeled.shape, dtype=bool)
#     for p in regionprops(green_filled_labeled):
#         comp = green_filled_labeled == p.label
#         if np.any(comp & target_mask):
#             n += 1
#             coloc[comp] = True
#     overlap_area = int((coloc & target_mask).sum())
#     return n, overlap_area, coloc
#
#
# print("Colocalization helpers defined.")

## 6 · Per‑cell analysis

In [7]:
# def analyze_cell(cid, labeled_cells, masks, raw):
#     """Per-cell measurements for one green-positive cell. Areas are pixel counts; MFI is
#     measured on the raw uint16 channels."""
#     cell_mask = labeled_cells == cid
#
#     # restrict every structure mask to this cell
#     green_struct = masks["green_struct"] & cell_mask
#     green_filled = masks["green_filled"] & cell_mask
#     red_in = masks["red"] & cell_mask       # lipid droplets
#     mito_in = masks["mito"] & cell_mask     # mitochondria
#
#     green_filled_lbl = label(green_filled)
#     n_green_struct = int(green_filled_lbl.max())  # dots + rings
#
#     # lipophagy (green/red) and mitophagy (green/blue), counted per green structure
#     n_gr, area_gr, _ = coloc_green_with(green_filled_lbl, red_in)
#     n_gb, area_gb, _ = coloc_green_with(green_filled_lbl, mito_in)
#
#     return {
#         "Cell_ID": cid,
#         "Cell_Area_px": int(cell_mask.sum()),
#         "N_Green_Structures": n_green_struct,
#         "Green_Structure_Area_px": int(green_struct.sum()),
#         "N_Lipid_Droplets": int(label(red_in).max()),
#         "LipidDroplet_Area_px": int(red_in.sum()),
#         "Mito_Area_px": int(mito_in.sum()),
#         "N_GreenRed_Coloc_Lipophagy": n_gr,
#         "GreenRed_Overlap_Area_px": area_gr,
#         "N_GreenBlue_Coloc_Mitophagy": n_gb,
#         "GreenBlue_Overlap_Area_px": area_gb,
#         "Green_MFI_raw": round(float(raw["green"][cell_mask].mean()), 3) if cell_mask.any() else 0.0,
#         "Yellow_MFI_raw": round(float(raw["yellow"][cell_mask].mean()), 3) if cell_mask.any() else 0.0,
#         "Red_MFI_raw": round(float(raw["red"][cell_mask].mean()), 3) if cell_mask.any() else 0.0,
#     }

In [8]:
# def analyze_cell(cid, labeled_cells, masks, raw):
#     """Per-cell measurements for one green-positive cell. Areas are pixel counts; MFI is
#     measured on the raw uint16 channels."""
#     cell_mask = labeled_cells == cid
#
#     # fill_green_rings already labeled and split these, so restrict and relabel rather than
#     # re-labeling a bool mask: 8-connectivity would diagonally re-merge what it just split.
#     green_objects = np.where(cell_mask, masks["green_objects"], 0)
#     green_objects, _, _ = relabel_sequential(green_objects)
#     ring_in = masks["green_ring_flag"] & cell_mask
#     red_in = masks["red"] & cell_mask       # lipid droplets
#     mito_in = masks["mito"] & cell_mask     # mitochondria
#
#     n_green_struct = int(green_objects.max())               # dots + rings
#     n_ring = sum(bool(np.any((green_objects == p.label) & ring_in))
#                  for p in regionprops(green_objects))        # rings / circles
#     n_dot = n_green_struct - n_ring                          # solid dots
#     green_area = int((green_objects > 0).sum())              # footprint (rings filled to disks)
#
#     # Counted per green structure; coloc_green_with takes the labels, so one streak over
#     # two rings counts as 2.
#     n_gr, area_gr, _ = coloc_green_with(green_objects, red_in)
#     n_gb, area_gb, _ = coloc_green_with(green_objects, mito_in)
#
#     return {
#         "Cell_ID": cid,
#         "Cell_Area_px": int(cell_mask.sum()),
#         "N_Green_Dots_Circles_Per_Cell": n_green_struct,
#         "N_Green_Structures": n_green_struct,
#         "N_Green_Dots": n_dot,
#         "N_Green_Rings": n_ring,
#         "Green_Dots_Circles_Area_px": green_area,
#         "Green_Structure_Area_px": green_area,
#         "N_Lipid_Droplets": int(label(red_in).max()),
#         "LipidDroplet_Area_px": int(red_in.sum()),
#         "Mito_Area_px": int(mito_in.sum()),
#         "N_GreenRed_Coloc_Lipophagy": n_gr,
#         "GreenRed_Overlap_Area_px": area_gr,
#         "N_GreenBlue_Coloc_Mitophagy": n_gb,
#         "GreenBlue_Overlap_Area_px": area_gb,
#         "Green_MFI_raw": round(float(raw["green"][cell_mask].mean()), 3) if cell_mask.any() else 0.0,
#         "Yellow_MFI_raw": round(float(raw["yellow"][cell_mask].mean()), 3) if cell_mask.any() else 0.0,
#         "Red_MFI_raw": round(float(raw["red"][cell_mask].mean()), 3) if cell_mask.any() else 0.0,
#     }

## 7 · Per‑image segmentation — `segment_one_image` builds + saves masks (per image file) and shows the QC figure

In [9]:
# cmap_g = LinearSegmentedColormap.from_list("G", [(0, 0, 0), (0, 1, 0)])
# cmap_r = LinearSegmentedColormap.from_list("R", [(0, 0, 0), (1, 0, 0)])
# cmap_y = LinearSegmentedColormap.from_list("Y", [(0, 0, 0), (1, 0.85, 0)])
# cmap_b = LinearSegmentedColormap.from_list("B", [(0, 0, 0), (0, 0.4, 1)])
#
#
# def show_mask(ax, mask, title, color="cyan"):
#     H, W = mask.shape
#     ax.imshow(np.zeros((H, W)), cmap="gray", vmin=0, vmax=1)
#     if mask.any():
#         ax.imshow(np.ma.masked_where(~mask, np.ones((H, W))),
#                   cmap=ListedColormap([color]), interpolation="none")
#     ax.set_title(title, color="white", fontsize=8)
#
#
# def show_mask_with_overlay(ax, base_mask, overlay_mask, title,
#                            base_color="lime", overlay_color="red"):
#     H, W = base_mask.shape
#     ax.imshow(np.zeros((H, W)), cmap="gray", vmin=0, vmax=1)
#     if base_mask.any():
#         ax.imshow(np.ma.masked_where(~base_mask, np.ones((H, W))),
#                   cmap=ListedColormap([base_color]), interpolation="none")
#     if overlay_mask.any():
#         ax.imshow(np.ma.masked_where(~overlay_mask, np.ones((H, W))),
#                   cmap=ListedColormap([overlay_color]), interpolation="none", alpha=0.9)
#     ax.set_title(title, color="white", fontsize=8)
#
#
# def save_image_masks(img_mask_dir, labeled_cells, masks, green_positive_ids):
#     """Save this image's masks into their own folder under mask_dir."""
#     os.makedirs(img_mask_dir, exist_ok=True)
#     np.save(os.path.join(img_mask_dir, "labeled_cells.npy"), labeled_cells)
#     np.save(os.path.join(img_mask_dir, "green_struct.npy"), masks["green_struct"])
#     np.save(os.path.join(img_mask_dir, "green_filled.npy"), masks["green_filled"])
#     np.save(os.path.join(img_mask_dir, "green_objects.npy"), masks["green_objects"])
#     np.save(os.path.join(img_mask_dir, "red_lipid.npy"), masks["red"])
#     np.save(os.path.join(img_mask_dir, "mito.npy"), masks["mito"])
#     np.save(os.path.join(img_mask_dir, "green_positive_ids.npy"),
#             np.array(green_positive_ids, dtype=np.int32))
#
#
# def show_qc_panels(filename, raw, labeled_cells, green_positive_ids, masks):
#     """QC figure: raw channels + segmentation + structure masks + colocalization."""
#     green_pos = np.isin(labeled_cells, green_positive_ids)
#     cells_bool = labeled_cells > 0
#     green_filled_lbl = np.where(cells_bool, masks["green_objects"], 0)  # keep the splits
#     _, _, gr_coloc = coloc_green_with(green_filled_lbl, masks["red"] & cells_bool)
#     _, _, gb_coloc = coloc_green_with(green_filled_lbl, masks["mito"] & cells_bool)
#
#     fig, ax = plt.subplots(3, 4, figsize=(22, 17))
#     fig.patch.set_facecolor("black")
#     fig.suptitle(filename, color="white", fontsize=11, y=0.995)
#
#     for a, im, cm, t in [
#         (ax[0, 0], boost_contrast_visual(raw["red"]), cmap_r, "Red (LipidTOX droplets)"),
#         (ax[0, 1], boost_contrast_visual(raw["yellow"]), cmap_y, "Yellow (mutant LC3)"),
#         (ax[0, 2], boost_contrast_visual(raw["green"]), cmap_g, "Green (normal LC3)"),
#         (ax[0, 3], boost_contrast_visual(raw["blue"]), cmap_b, "Blue (mitochondria)")]:
#         a.imshow(im, cmap=cm); a.set_title(t, color="white", fontsize=9)
#
#     ax[1, 0].imshow(label2rgb(labeled_cells, bg_label=0, bg_color=(0, 0, 0)), interpolation="none")
#     ax[1, 0].set_title(f"1. Cell mask — {int(labeled_cells.max())} cells", color="white", fontsize=8)
#     gp_labels = np.where(green_pos, labeled_cells, 0)
#     ax[1, 1].imshow(label2rgb(gp_labels, bg_label=0, bg_color=(0, 0, 0)), interpolation="none")
#     ax[1, 1].set_title(f"2. Green-positive cells — {len(green_positive_ids)}", color="white", fontsize=8)
#     show_mask(ax[1, 2], masks["green_struct"], "3. Green dots + ring walls", "lime")
#     show_mask(ax[1, 3], masks["green_filled"], "4. Green filled (rings→disks)", "lime")
#
#     show_mask(ax[2, 0], masks["red"], "5. Red lipid droplets", "red")
#     show_mask(ax[2, 1], masks["mito"], "6. Mitochondria (streaks)", "deepskyblue")
#     show_mask_with_overlay(ax[2, 2], gr_coloc, masks["red"] & cells_bool,
#                            "7. Lipophagy: green∩red", "lime", "red")
#     show_mask_with_overlay(ax[2, 3], gb_coloc, masks["mito"] & cells_bool,
#                            "8. Mitophagy: green∩blue", "lime", "deepskyblue")
#
#     for a in ax.ravel():
#         a.set_facecolor("black")
#         a.set_xticks([]); a.set_yticks([])
#         for spine in a.spines.values():
#             spine.set_visible(True); spine.set_color("white"); spine.set_linewidth(0.6)
#     plt.tight_layout()
#     plt.show()
#
#
# def segment_one_image(filepath, mask_dir):
#     """Build and save every mask for one .czi, and return the bundle the analysis needs.
#
#     None if the file cannot be loaded or contains no cells."""
#     filename = os.path.basename(filepath)
#     stem = os.path.splitext(filename)[0]
#     print(f"\nProcessing: {filename}")
#
#     ch = load_four_channels(filepath)
#     if ch is None:
#         print("  Skipping (load error)."); return None
#     raw = ch
#     norm = {k: normalize_image(v) for k, v in ch.items()}  # min-max -> inclusion analysis
#     seg = {k: pct_norm(v) for k, v in ch.items()}          # percentile -> segmentation
#
#     # Cell segmentation. No nucleus channel, so cells come from the LC3 fill.
#     if USE_CELLPOSE:
#         labeled_cells = segment_whole_cells_cellpose(seg)
#     else:
#         labeled_cells = segment_cells_intensity(seg)
#     if labeled_cells is None or labeled_cells.max() == 0:
#         print("  No cells found, skipping."); return None
#
#     green_positive_ids, fills = classify_green_positive(labeled_cells, seg["green"])
#     print(f"    Green-positive cells: {len(green_positive_ids)} / {labeled_cells.max()} "
#           f"(fill >= {GREEN_FILL_FRAC})")
#
#     # Inclusion-analysis masks, off the min-max `norm` dict.
#     green_struct = detect_green_structures(norm["green"], labeled_cells)
#     green_objects, green_ring_flag = fill_green_rings(green_struct)
#     green_filled = green_objects > 0                                  # kept for QC and saving
#     red_mask = detect_bright_puncta(norm["red"], RED_TOPHAT_RADIUS, RED_PCTL_THRESH,
#                                     RED_MIN_SIZE, RED_MAX_SIZE)
#     red_mask &= labeled_cells > 0
#     mito_mask = detect_mitochondria(norm["blue"], labeled_cells)
#
#     masks = {"green_struct": green_struct, "green_objects": green_objects,
#              "green_ring_flag": green_ring_flag, "green_filled": green_filled,
#              "red": red_mask, "mito": mito_mask}
#
#     img_mask_dir = os.path.join(mask_dir, stem)
#     save_image_masks(img_mask_dir, labeled_cells, masks, green_positive_ids)
#     print(f"    saved masks -> {img_mask_dir}")
#
#     if SHOW_PLOTS:
#         show_qc_panels(filename, raw, labeled_cells, green_positive_ids, masks)
#
#     return {"labeled_cells": labeled_cells, "masks": masks,
#             "raw": {"green": raw["green"], "yellow": raw["yellow"], "red": raw["red"]},
#             "green_positive_ids": green_positive_ids}
#
#
# print("Per-image segmentation defined (driven from the sequential driver below).")

## 8 · Sequential driver — for each folder: segment every image → save masks → analyze green‑positive cells → write `<folder>_…analysis.csv`

In [10]:
# def analyze_folder(image_folder):
#     """Segment and analyze every image in one folder.
#
#     Masks are saved per image under Results/masks/<folder>/<image>/ and one CSV is written
#     for the folder."""
#     folder_tag = os.path.basename(os.path.normpath(image_folder))
#     mask_dir = os.path.join(MASK_FOLDER, folder_tag)
#     os.makedirs(mask_dir, exist_ok=True)
#
#     image_files = sorted(glob.glob(os.path.join(image_folder, FILE_EXTENSION)))
#     print(f"{folder_tag}: {len(image_files)} image(s)")
#
#     rows = []
#     for filepath in image_files:
#         result = segment_one_image(filepath, mask_dir)
#         if result is None:
#             continue
#         for cid in result["green_positive_ids"]:
#             row = analyze_cell(cid, result["labeled_cells"], result["masks"], result["raw"])
#             row["Filename"] = os.path.basename(filepath)
#             rows.append(row)
#
#     if rows:
#         df = pd.DataFrame(rows)
#         front = ["Filename", "Cell_ID"]
#         df = df[front + [c for c in df.columns if c not in front]]
#     else:
#         df = pd.DataFrame()
#
#     out_csv = os.path.join(RESULTS_FOLDER, f"{folder_tag}_{CSV_SUFFIX}")
#     df.to_csv(out_csv, index=False)
#     print(f"  -> {len(df)} green-positive cell row(s) saved to {out_csv}\n")
#     return df
#
#
# last_df = None
# for image_folder in IMAGE_FOLDERS:
#     folder_tag = os.path.basename(os.path.normpath(image_folder))
#     if not os.path.isdir(image_folder):
#         print(f"[skip] '{image_folder}' not found\n")
#         continue
#     print(f"{folder_tag}: START")
#     last_df = analyze_folder(image_folder)
#
# print("All folders processed — one CSV each in Results/, "
#       "masks in Results/masks/<folder>/<image>/.")
# last_df.head() if last_df is not None else None

## 9 · Output summary — list the per‑folder CSVs and where the per‑image masks were saved

In [11]:
# csv_files = sorted(glob.glob(os.path.join(RESULTS_FOLDER, f"*_{CSV_SUFFIX}")))
# print(f"{len(csv_files)} CSV file(s) in {RESULTS_FOLDER}/:")
# total_rows = 0
# for f in csv_files:
#     n = len(pd.read_csv(f))
#     total_rows += n
#     print(f"  {os.path.basename(f):70s} {n:>4d} green-positive cell row(s)")
# print(f"  {'TOTAL':70s} {total_rows:>4d}")
#
# print(f"\nMasks (per image file) under {MASK_FOLDER}/<folder>/<image>/:")
# print("  labeled_cells.npy, green_struct.npy, green_filled.npy,")
# print("  red_lipid.npy, mito.npy, green_positive_ids.npy")  

## 10 · CSV output — additional analysis from the saved masks

Reads `Results/masks/<image>/*.npy` (no re-segmentation) and writes two CSVs:

| File | Contents |
|---|---|
| `Additional_Analysis_per_cell.csv` | one row per green-positive cell |
| `Additional_Analysis_summary_by_condition.csv` | n, mean and sample SD per metric, CH vs noCH |

The original per-cell analysis is unchanged and stays in
`L3 Lipophagy images_LC3_lipophagy_mitophagy_analysis.csv`.

New columns: green circles surrounding lipid droplets, surrounding mitochondria, and
surrounding either; cytoplasmic surface area; lipid droplet count; lipid droplet and
mitochondrial surface area. All areas are pixel counts.

A green object counts as a *circle* when the ring fill enclosed an interior for it, and as
*surrounding* a target when that interior contains target pixels. That is stricter than the
`N_GreenRed_Coloc_Lipophagy` / `N_GreenBlue_Coloc_Mitophagy` columns of the original
analysis, which count any green object merely touching a target, solid dots included.

The cell asserts that every column it shares with the original CSV reproduces it exactly
before writing anything.

In [12]:
import os

import numpy as np
import pandas as pd
from skimage.measure import label, regionprops
from skimage.segmentation import relabel_sequential

RESULTS_FOLDER = "Results"
IMAGE_FOLDER_NAME = "L3 Lipophagy images"
MASK_ROOT = os.path.join(RESULTS_FOLDER, "masks", IMAGE_FOLDER_NAME)
SOURCE_CSV = os.path.join(RESULTS_FOLDER, f"{IMAGE_FOLDER_NAME}_LC3_lipophagy_mitophagy_analysis.csv")
CSV_STEM = os.path.join(RESULTS_FOLDER, "Additional_Analysis")


def measure_cell(cell_mask, green_objects, green_struct, red, mito):
    """Per-cell counts for one green-positive cell, read back from the saved masks.

    A green object is a circle when fill_green_rings enclosed an interior for it, and
    surrounds a target when that interior holds target pixels."""
    objs = np.where(cell_mask, green_objects, 0)
    objs, _, _ = relabel_sequential(objs)       # honour the watershed splits
    red_in = red & cell_mask
    mito_in = mito & cell_mask

    n_circles = n_ld = n_mito = n_either = 0
    for p in regionprops(objs):
        lumen = (objs == p.label) & ~green_struct    # empty for a solid dot
        if not lumen.any():
            continue
        n_circles += 1
        around_ld = bool((red_in & lumen).any())
        around_mito = bool((mito_in & lumen).any())
        n_ld += around_ld
        n_mito += around_mito
        n_either += around_ld or around_mito

    return {
        "Cytoplasm_SA_px": int(cell_mask.sum()),
        "N_Green_Circles": n_circles,
        "N_Green_Circles_Surrounding_LD": n_ld,
        "N_Green_Circles_Surrounding_Mito": n_mito,
        "N_Green_Circles_Surrounding_LD_or_Mito": n_either,
        "N_Lipid_Droplets": int(label(red_in).max()),
        "LipidDroplet_SA_px": int(red_in.sum()),
        "Mito_SA_px": int(mito_in.sum()),
    }


def collect_from_masks(mask_root):
    rows = []
    for stem in sorted(os.listdir(mask_root)):
        image_dir = os.path.join(mask_root, stem)
        if not os.path.isdir(image_dir):
            continue

        def load(name):
            return np.load(os.path.join(image_dir, name))

        labeled_cells = load("labeled_cells.npy")
        green_objects = load("green_objects.npy")
        green_struct = load("green_struct.npy")
        red = load("red_lipid.npy")
        mito = load("mito.npy")
        for cid in load("green_positive_ids.npy"):
            row = {"Filename": f"{stem}.czi", "Cell_ID": int(cid)}
            row.update(measure_cell(labeled_cells == cid, green_objects, green_struct, red, mito))
            rows.append(row)
    return pd.DataFrame(rows)


# --- measure ---------------------------------------------------------------

new = collect_from_masks(MASK_ROOT)
original = pd.read_csv(SOURCE_CSV)

# The masks must still reproduce the shipped CSV, or these columns describe other cells.
check = original.merge(new, on=["Filename", "Cell_ID"], suffixes=("_orig", "_new"))
assert len(check) == len(original) == len(new), "mask rows do not line up with the CSV"
for a, b in [("Cell_Area_px", "Cytoplasm_SA_px"),
             ("N_Green_Rings", "N_Green_Circles"),
             ("N_Lipid_Droplets_orig", "N_Lipid_Droplets_new"),
             ("LipidDroplet_Area_px", "LipidDroplet_SA_px"),
             ("Mito_Area_px", "Mito_SA_px")]:
    assert (check[a] == check[b]).all(), f"{a} != {b}"
print(f"mask readback matches {os.path.basename(SOURCE_CSV)} on all shared columns")

new.insert(1, "Condition", np.where(new["Filename"].str.contains("_noCH_"), "noCH", "CH"))
new["Condition"] = pd.Categorical(new["Condition"], ["CH", "noCH"], ordered=True)
new = new.sort_values(["Condition", "Filename", "Cell_ID"]).reset_index(drop=True)
new["Condition"] = new["Condition"].astype(str)

per_cell = new[["Filename", "Condition", "Cell_ID",
                "Cytoplasm_SA_px",
                "N_Green_Circles",
                "N_Green_Circles_Surrounding_LD",
                "N_Green_Circles_Surrounding_Mito",
                "N_Green_Circles_Surrounding_LD_or_Mito",
                "N_Lipid_Droplets",
                "LipidDroplet_SA_px",
                "Mito_SA_px"]]

# --- per-cell table ---------------------------------------------------------

per_cell.to_csv(f"{CSV_STEM}_per_cell.csv", index=False)

# --- summary by condition ---------------------------------------------------

metrics = [c for c in per_cell.columns if c not in ("Filename", "Condition", "Cell_ID")]
summary = []
for metric in metrics:
    row = {"Metric": metric}
    for cond in ("CH", "noCH"):
        values = per_cell.loc[per_cell["Condition"] == cond, metric]
        row[f"{cond}_n_cells"] = int(values.count())
        row[f"{cond}_mean"] = round(float(values.mean()), 2)
        row[f"{cond}_SD"] = round(float(values.std(ddof=1)), 2)
    summary.append(row)
pd.DataFrame(summary).to_csv(f"{CSV_STEM}_summary_by_condition.csv", index=False)

for suffix in ("per_cell", "summary_by_condition"):
    print(f"wrote {CSV_STEM}_{suffix}.csv")


mask readback matches L3 Lipophagy images_LC3_lipophagy_mitophagy_analysis.csv on all shared columns
wrote Results/Additional_Analysis_per_cell.csv
wrote Results/Additional_Analysis_summary_by_condition.csv
